# GLiNER2 on a T4: the animations

Each clip makes one claim, in one sentence. They share one visual system (`style.py`): a fixed palette (CPU blue,
GPU orange, wasted work grey, correct green, error red), Inter for labels, JetBrains Mono for numbers, a charcoal
background, and slow eased motion with one thing moving at a time.

The clips are silent but narrated on screen: each is split into chapters (top left) and carries a narration line at
the bottom that changes beat by beat, so a viewer can follow without a voiceover. They run 65–90 s. The method,
measured from 3Blue1Brown's own scene code, is written up as a reusable skill in
`.claude/skills/explainer-animations/` in this repository.

Runs on a Colab CPU runtime; rendering a clip at 1080p takes a few minutes.

In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    !apt-get -qq install -y libcairo2-dev libpango1.0-dev ffmpeg fonts-inter fonts-jetbrains-mono > /dev/null 2>&1 || true
    !pip -q install manim
fonts = Path.home() / ".fonts"
fonts.mkdir(exist_ok=True)
for name, url in [("Inter.ttf", "https://github.com/google/fonts/raw/main/ofl/inter/Inter%5Bopsz,wght%5D.ttf"),
                  ("JetBrainsMono.ttf", "https://github.com/google/fonts/raw/main/ofl/jetbrainsmono/JetBrainsMono%5Bwght%5D.ttf")]:
    if not (fonts / name).exists():
        !wget -q -O {fonts / name} "{url}"
!fc-cache -f > /dev/null && fc-list : family | grep -E "^(Inter|JetBrains Mono)$" | sort -u

## The visual system

In [ ]:
%%writefile style.py
# The visual system for explainer clips (Manim Community Edition). Every clip imports from here and never defines its
# own colours, fonts or timing. Copy this file next to the clips; see ../SKILL.md for how to use it.
from manim import *

# Surfaces and ink
BACKGROUND = "#1C1E21"
PANEL = "#26292E"
PANEL_EDGE = "#3A3F45"
INK = "#E8E6E1"
INK_DIM = "#8B9096"
INK_FAINT = "#4A4F55"

# Colours with fixed meanings: never reuse one for something else within a project
CPU = "#58C4DD"
GPU = "#FF9F43"
WASTE = "#6B7178"
CORRECT = "#83C167"
ERROR = "#FC6255"
NUMBER = "#FFD84D"
TERM_C2C = "#5CD0B3"
TERM_C2P = "#C792EA"
TERM_P2C = "#FF8FB1"
LABEL_COLORS = {"Person": "#58C4DD", "Organization": "#C792EA", "Street": "#83C167", "City": "#FF9F43"}

SANS = "Inter"
MONO = "JetBrains Mono"

# Timing, measured from 3b1b's 2024 transformer scenes: median animation 2 s, a pause after most animations
BEAT = 2.0
PAUSE = 1.0
HOLD = 2.5
READ_WORDS_PER_SECOND = 3.0

config.background_color = BACKGROUND
config.frame_rate = 30


def words(text, size=28, color=INK, weight=NORMAL, font=SANS):
    return Text(text, font=font, font_size=size, color=color, weight=weight)


def mono(text, size=26, color=INK):
    return Text(text, font=MONO, font_size=size, color=color)


def wrap(text, line_len=72):
    lines, line = [], ""
    for word in text.split():
        if line and len(line) + 1 + len(word) > line_len:
            lines.append(line)
            line = word
        else:
            line = f"{line} {word}".strip()
    return "\n".join(lines + [line])


def span(text_mob, source, phrase, occurrence=0):
    # the glyphs of `phrase` inside a Text built from `source` (Text keeps no glyphs for whitespace)
    start = -1
    for _ in range(occurrence + 1):
        start = source.index(phrase, start + 1)
    first = len("".join(source[:start].split()))
    return text_mob[first:first + len("".join(phrase.split()))]


def box(mob, color=INK_DIM, fill=0.18, buff=0.07, stroke=2):
    rect = SurroundingRectangle(mob, buff=buff, corner_radius=0.06)
    rect.set_stroke(color, stroke).set_fill(color, fill)
    return rect


def panel(width, height, fill=PANEL, edge=PANEL_EDGE):
    return RoundedRectangle(width=width, height=height, corner_radius=0.18).set_fill(fill, 1).set_stroke(edge, 1.5)


def chip(label, color, size=22):
    text = words(label, size=size, color=color, weight=MEDIUM)
    back = RoundedRectangle(width=text.width + 0.36, height=text.height + 0.26, corner_radius=0.14)
    back.set_fill(color, 0.14).set_stroke(color, 1.5)
    return VGroup(back, text.move_to(back))


def chip_level(group, level):
    chips = [group] if isinstance(group[0], RoundedRectangle) else list(group)
    return AnimationGroup(*(AnimationGroup(c[0].animate.set_fill(opacity=0.14 * level).set_stroke(opacity=level),
                                           c[1].animate.set_opacity(level)) for c in chips))


def caption(sentence, size=34):
    line = words(sentence, size=size, color=INK, weight=MEDIUM)
    if line.width > config.frame_width - 1.2:
        line.scale_to_fit_width(config.frame_width - 1.2)
    return line.to_edge(DOWN, buff=0.55)


def bracket(left, right, y, color=INK_DIM, tick=0.12, up=True, stroke=2.5):
    sign = -1 if up else 1
    left, right = np.array([left, y, 0]), np.array([right, y, 0])
    return VGroup(Line(left, right), Line(left, left + sign * tick * UP), Line(right, right + sign * tick * UP)).set_stroke(
        color, stroke)


def hatch(rect, spacing=0.14, color=WASTE, stroke=1.6):
    x0, x1 = rect.get_left()[0], rect.get_right()[0]
    y0, y1 = rect.get_bottom()[1], rect.get_top()[1]
    lines = VGroup()
    offset = x0 - (y1 - y0)
    while offset < x1:
        start = np.array([offset, y0, 0])
        end = start + (y1 - y0) * np.array([1, 1, 0])
        if start[0] < x0:
            start = start + (x0 - start[0]) * np.array([1, 1, 0])
        if end[0] > x1:
            end = end - (end[0] - x1) * np.array([1, 1, 0])
        if end[0] > start[0]:
            lines.add(Line(start, end))
        offset += spacing
    return lines.set_stroke(color, stroke)


def counter(tracker, size=60, color=NUMBER, fmt="{:.0f}"):
    # animate only the tracker; never animate the counter itself (it is rebuilt every frame)
    paint = color if callable(color) else (lambda: color)
    return always_redraw(lambda: words(fmt.format(tracker.get_value()), size=size, color=paint(), weight=BOLD,
                                       font=MONO))


def heat_grid(values, color, cell=0.42, gap=0.04):
    grid = VGroup()
    rows, cols = len(values), len(values[0])
    for i in range(rows):
        for j in range(cols):
            square = Square(cell).set_stroke(width=0).set_fill(color, 0.08 + 0.85 * values[i][j])
            square.move_to([(j - (cols - 1) / 2) * (cell + gap), -(i - (rows - 1) / 2) * (cell + gap), 0])
            grid.add(square)
    return grid


def connector(small, big, color=INK_DIM, stroke=1.5):
    # 3b1b's link between a symbol and its concrete expansion: two lines from the small box's corners to the big box's
    if small.get_center()[1] > big.get_center()[1]:
        pairs = [(small.get_corner(DL), big.get_corner(UL)), (small.get_corner(DR), big.get_corner(UR))]
    else:
        pairs = [(small.get_corner(UL), big.get_corner(DL)), (small.get_corner(UR), big.get_corner(DR))]
    return VGroup(*(Line(a, b) for a, b in pairs)).set_stroke(color, stroke)


class Derive(Transform):
    """TransformFromCopy that ends with `target` itself on screen. Transform pads the target's submobjects to match the
    source, which breaks glyph indexing (span) on Text targets; transforming copies and swapping in the original keeps
    the target intact."""

    def __init__(self, source, target, **kwargs):
        self.final = target
        super().__init__(source.copy(), target.copy(), **kwargs)

    def clean_up_from_scene(self, scene):
        super().clean_up_from_scene(scene)
        scene.remove(self.mobject)
        scene.add(self.final)


class Clip(Scene):
    """A silent explainer: chapters top left, a narration line at the bottom that the viewer reads."""

    def setup(self):
        self.narration = None
        self.narration_start = 0.0
        self.chapter_title = None

    def beat(self, *animations, run_time=BEAT, **kwargs):
        self.play(*animations, run_time=run_time, rate_func=kwargs.pop("rate_func", smooth), **kwargs)

    def pause(self, seconds=PAUSE):
        self.wait(seconds)

    def chapter(self, number, total, title):
        label = VGroup(words(f"{number} / {total}", size=18, color=INK_FAINT, font=MONO),
                       words(title, size=26, color=INK, weight=MEDIUM)).arrange(RIGHT, buff=0.3)
        label.to_corner(UL, buff=0.55)
        if self.chapter_title is None:
            self.play(FadeIn(label, shift=0.15 * DOWN), run_time=0.8)
        else:
            self.play(FadeOut(self.chapter_title, shift=0.15 * UP), FadeIn(label, shift=0.15 * UP), run_time=0.8)
        self.chapter_title = label

    def say(self, text, *animations, run_time=None):
        """Replace the narration line, together with any animations that illustrate it."""
        line = words(wrap(text), size=25, color=INK).to_edge(DOWN, buff=0.45)
        if self.narration is not None:
            self.play(FadeOut(self.narration, shift=0.1 * UP), run_time=0.35)
        self.play(FadeIn(line, shift=0.12 * UP), *animations,
                  run_time=run_time or (BEAT if animations else 0.8), rate_func=smooth)
        self.narration = line
        self.narration_start = self.renderer.time - (run_time or (BEAT if animations else 0.8))

    def read(self, extra=0.0):
        """Wait until the current narration line has been on screen long enough to read, plus a pause."""
        if self.narration is None:
            self.wait(PAUSE + extra)
            return
        needed = len(self.narration.text.split()) / READ_WORDS_PER_SECOND + 0.6
        elapsed = self.renderer.time - self.narration_start
        self.wait(max(PAUSE * 0.6, needed - elapsed) + extra)

    def unsay(self):
        if self.narration is not None:
            self.play(FadeOut(self.narration), run_time=0.5)
            self.narration = None

    def finale(self, sentence, note=None, size=34):
        self.unsay()
        claim = caption(sentence, size=size)
        parts = [FadeIn(claim, shift=0.2 * UP)]
        if note:
            parts.append(FadeIn(words(note, size=20, color=INK_DIM).next_to(claim, DOWN, buff=0.12)))
        self.play(*parts, run_time=1.2)
        self.wait(HOLD)
        self.play(*(FadeOut(mob) for mob in self.mobjects), run_time=1.0)

In [ ]:
from IPython.display import Video, display


def render(source, scene):
    !manim -qh --disable_caching {source} {scene}
    video = f"media/videos/{Path(source).stem}/1080p30/{scene}.mp4"
    display(Video(video, embed=True, width=960))
    return video


def gif(video, out):
    !ffmpeg -v error -y -i {video} -vf "fps=12,scale=640:-1:flags=lanczos,palettegen=stats_mode=diff" palette.png
    !ffmpeg -v error -y -i {video} -i palette.png -lavfi "fps=12,scale=640:-1:flags=lanczos[x];[x][1:v]paletteuse=dither=bayer:bayer_scale=5:diff_mode=rectangle" {out}
    !ls -la {out}

## A. Read once, ask everything

> Instead of reading the document three times, read it once and ask all the questions.

In [ ]:
%%writefile clip_a_read_once.py
# Clip A: "Instead of reading the document three times, read it once and ask all the questions."
# Chapters: 1 the task (a line of a contract, four labels, the output we want); 2 how GLiNER2 reads: labels go into the
# prompt ahead of the text, the encoder turns every word into a vector (the expensive step), a light step matches labels
# to words; 3 before: three label groups, so the encoder reads the same text three times; 4 after: one prompt with every
# label, one encoder run, 2.4x the throughput with per-label thresholds.
# Numbers: Tesla T4, realistic corpus (290 chunks), round 8 of benchmark_results/pipeline_2026-10-25: 3 passes
# 59.0 chunks/s, single pass with per-label thresholds (Organization 0.85, Street 0.5) 142.2 chunks/s, no label worse.
from style import *

SOURCE = "Lakshmi Varma of Kaveri Analytics, 114 Malabar Lane, Kochi"
TOKENS = ["Lakshmi", "Varma", "of", "Kaveri", "Analytics", "114", "Malabar", "Lane", "Kochi"]
ENTITIES = [("Person", 0, 2), ("Organization", 3, 5), ("Street", 5, 8), ("City", 8, 9)]
OLD_PASSES = [["Organization"], ["Person", "Street", "City"], ["City"]]
LABELS = ["Person", "Organization", "Street", "City"]


def token(text):
    label = words(text, size=20, color=INK)
    back = RoundedRectangle(width=label.width + 0.24, height=0.46, corner_radius=0.08)
    back.set_fill(PANEL, 1).set_stroke(PANEL_EDGE, 1.5)
    return VGroup(back, label.move_to(back))


class ReadOnce(Clip):
    def construct(self):
        self.the_task()
        self.how_gliner_reads()
        self.three_passes()
        self.one_pass()
        self.finale("Read the document once, and ask all the questions.",
                    "Tesla T4 · realistic corpus · 59 → 142 chunks/s, no label worse")

    def the_task(self):
        self.chapter(1, 4, "The task")
        line = words(SOURCE, size=34, color=INK).move_to(UP * 1.6)
        self.say("A line from a contract. We want to find the personal data in it.",
                 LaggedStart(*(FadeIn(ch, shift=0.1 * UP) for ch in line), lag_ratio=0.02))
        self.read()
        chips = VGroup(*(chip(l, LABEL_COLORS[l]) for l in LABELS)).arrange(RIGHT, buff=0.18).move_to(UP * 2.55)
        self.say("Four kinds: people, organizations, streets and cities.",
                 LaggedStart(*(FadeIn(c, shift=0.15 * DOWN) for c in chips), lag_ratio=0.2))
        self.read()
        phrases = {"Person": "Lakshmi Varma", "Organization": "Kaveri Analytics", "Street": "114 Malabar Lane",
                   "City": "Kochi"}
        boxes = VGroup(*(box(span(line, SOURCE, phrases[l]), LABEL_COLORS[l], fill=0.2) for l in LABELS))
        self.say("This is the answer we want: every entity found, with the right label.",
                 LaggedStart(*(DrawBorderThenFill(b) for b in boxes), lag_ratio=0.25))
        self.read()

        tokens = VGroup(*(token(t) for t in TOKENS)).arrange(RIGHT, buff=0.1).move_to(UP * 1.6)
        self.tokens = tokens
        pieces = [span(line, SOURCE, t) for t in TOKENS]
        self.say("The model reads text as a row of pieces, roughly one per word.",
                 FadeOut(boxes), FadeOut(line), *(Derive(p, t) for p, t in zip(pieces, tokens)), run_time=2.2)
        self.read()
        self.chips = chips

    def how_gliner_reads(self):
        self.chapter(2, 4, "How GLiNER2 reads")
        tokens = self.tokens
        anchored = tokens.copy().to_edge(RIGHT, buff=0.65)
        prompt = VGroup(chip("Organization", LABEL_COLORS["Organization"], size=19)).next_to(anchored, LEFT, buff=0.35)
        self.say("The labels we ask for are written into the input, in front of the text.",
                 tokens.animate.move_to(anchored), Derive(self.chips[1], prompt[0]), FadeOut(self.chips), run_time=2.2)
        self.prompt = prompt
        self.read()

        encoder = RoundedRectangle(width=10.6, height=1.05, corner_radius=0.16).set_fill(GPU, 0.1).set_stroke(GPU, 2.5)
        encoder.move_to(UP * 0.15)
        encoder_label = words("encoder: mDeBERTa, 12 layers", size=22, color=GPU, weight=MEDIUM).move_to(encoder)
        self.encoder = VGroup(encoder, encoder_label)
        self.say("The encoder reads the whole input at once. It is the expensive part: almost all the GPU time.",
                 FadeIn(self.encoder, shift=0.1 * UP))
        self.read()
        vectors = self.encode(VGroup(prompt, tokens))
        self.say("Out comes one vector per piece: what each word means here, with the labels in view.", run_time=1.0)
        self.read()

        match = box(VGroup(*tokens[3:5]), LABEL_COLORS["Organization"], fill=0.22)
        arc = ArcBetweenPoints(prompt.get_top(), match.get_top(), angle=-PI / 3)
        arc.set_stroke(LABEL_COLORS["Organization"], 2.5)
        self.say("A light step then scores each label against the word vectors, and marks the matches.",
                 Create(arc), DrawBorderThenFill(match), run_time=2.2)
        self.read()
        self.beat(FadeOut(VGroup(arc, match, vectors)), run_time=0.8)

    def encode(self, row, run_time=1.6):
        drops = VGroup(*(m.copy() for m in row))
        bars = VGroup()
        for n, t in enumerate(self.tokens):
            bar = VGroup(*(Square(0.13).set_stroke(width=0).set_fill(GPU, 0.3 + 0.65 * (((n * 7 + k * 3) % 7) / 6))
                           for k in range(5))).arrange(DOWN, buff=0.035)
            bars.add(bar.next_to(self.encoder, DOWN, buff=0.28).match_x(t))
        self.beat(LaggedStart(*(m.animate.move_to(self.encoder).set_opacity(0) for m in drops), lag_ratio=0.05),
                  self.encoder[0].animate.set_fill(GPU, 0.45), run_time=run_time)
        self.remove(drops)
        self.beat(self.encoder[0].animate.set_fill(GPU, 0.1),
                  LaggedStart(*(GrowFromEdge(b, UP) for b in bars), lag_ratio=0.05), run_time=run_time * 0.8)
        return bars

    def meter(self):
        title = words("encoder runs", size=20, color=INK_DIM).move_to(LEFT * 4.6 + DOWN * 2.0, aligned_edge=LEFT)
        rate_title = words("throughput", size=20, color=INK_DIM).move_to(RIGHT * 0.9 + DOWN * 2.0, aligned_edge=LEFT)
        self.runs = ValueTracker(0)
        count = counter(self.runs, size=40, color=GPU, fmt="×{:.0f}")
        count.add_updater(lambda m: m.next_to(title, RIGHT, buff=0.25))
        self.rate = ValueTracker(0)
        rate = counter(self.rate, size=40, color=NUMBER)
        unit = words("chunks/s", size=18, color=INK_DIM)
        rate.add_updater(lambda m: m.next_to(rate_title, RIGHT, buff=0.25))
        unit.add_updater(lambda m: m.next_to(rate, RIGHT, buff=0.15).align_to(rate, DOWN))
        for m in (count, rate, unit):
            m.update()
        return title, count, rate_title, rate, unit

    def place_prompt(self, prompt):
        if len(prompt) > 2:
            prompt.arrange_in_grid(cols=2, buff=(0.08, 0.08), cell_alignment=RIGHT)
        return prompt.next_to(self.tokens, LEFT, buff=0.35)

    def three_passes(self):
        self.chapter(3, 4, "Before: three passes")
        tokens = self.tokens
        title, count, rate_title, rate, unit = self.meter()
        self.say("The old setup asked for the labels in three groups.", FadeIn(title))
        self.add(count)
        self.read()
        found, boxes = {}, VGroup()
        texts = ["Pass 1: ask for Organization.", "Pass 2: the same text again, for the other three labels.",
                 "Pass 3: the same text a third time, for cities alone."]
        for n, group in enumerate(OLD_PASSES):
            prompt = VGroup(*(chip(l, LABEL_COLORS[l], size=19) for l in group)).arrange(RIGHT, buff=0.08)
            self.place_prompt(prompt)
            self.say(texts[n], FadeOut(self.prompt, shift=0.2 * UP), FadeIn(prompt, shift=0.2 * UP), run_time=1.4)
            self.prompt = prompt
            vectors = self.encode(VGroup(prompt, tokens), run_time=1.6 if n == 0 else 1.2)
            self.beat(self.runs.animate.set_value(n + 1), run_time=0.6)
            marks = []
            for label, start, end in ENTITIES:
                if label not in group:
                    continue
                if label in found:
                    marks.append(found[label].animate.set_stroke(WASTE).set_fill(WASTE, 0.25))
                else:
                    found[label] = box(VGroup(*tokens[start:end]), LABEL_COLORS[label], fill=0.22)
                    boxes.add(found[label])
                    marks.append(DrawBorderThenFill(found[label]))
            self.beat(*marks, FadeOut(vectors), run_time=1.2)
            self.read()
        self.say("Same text, encoded three times. The third pass found nothing new.", run_time=1.0)
        self.read()
        self.add(rate)
        self.say("That setup ran at 59 chunks per second on one T4.",
                 FadeIn(rate_title), FadeIn(unit), self.rate.animate.set_value(59), run_time=2.0)
        self.read()
        self.old_boxes = boxes

    def one_pass(self):
        self.chapter(4, 4, "After: one pass")
        tokens = self.tokens
        prompt = VGroup(*(chip(l, LABEL_COLORS[l], size=19) for l in LABELS)).arrange(RIGHT, buff=0.08)
        self.place_prompt(prompt)
        self.say("Now every label goes into one prompt.",
                 FadeOut(self.prompt, shift=0.2 * UP), FadeIn(prompt, shift=0.2 * UP), FadeOut(self.old_boxes),
                 self.runs.animate.set_value(0), run_time=2.0)
        self.prompt = prompt
        self.read()
        vectors = self.encode(VGroup(prompt, tokens), run_time=1.8)
        self.beat(self.runs.animate.set_value(1), run_time=0.6)
        boxes = VGroup(*(box(VGroup(*tokens[s:e]), LABEL_COLORS[l], fill=0.22) for l, s, e in ENTITIES))
        self.say("One encoder run. The light matching step answers all four questions from the same vectors.",
                 LaggedStart(*(DrawBorderThenFill(b) for b in boxes), lag_ratio=0.2), FadeOut(vectors), run_time=2.2)
        self.read()
        self.say("Throughput rose from 59 to 142 chunks per second: 2.4 times as fast.",
                 self.rate.animate.set_value(142), run_time=2.4)
        self.read()
        note = words("thresholds per label (Organization 0.85, Street 0.5): every label as accurate or better",
                     size=19, color=INK_DIM).move_to(DOWN * 1.0)
        self.say("Accuracy held: with a threshold per label, no label got worse on our test corpus.",
                 FadeIn(note, shift=0.1 * UP))
        self.read(extra=0.5)

In [ ]:
video = render("clip_a_read_once.py", "ReadOnce")
gif(video, "clip_a_read_once.gif")

## B. The CPU wasn't working, it was waiting

> The slow line wasn't slow: it was where the CPU waited for the GPU.

In [ ]:
%%writefile clip_b_waiting.py
# Clip B: "The slow line wasn't slow: it was where the CPU waited for the GPU."
# Chapters: 1 the profile (four lines of serving code; cProfile blames .tolist() for half the time); 2 two workers
# (launching the encoder only queues work on the GPU and returns); 3 .tolist() is waiting (the line that needs the
# numbers blocks until the GPU is done, the profiler books that wait to it; between batches the GPU sits idle);
# 4 the fix (prepare and launch the next batch before asking for this one's result: the GPU's gaps close).
# Numbers: Tesla T4, benchmark_results/pipeline_2026-10-25/README.md: cProfile of the queue path, .tolist() 2.538 s =
# 50.3% of profiled time (round 5), 0.778 s with the one-thread pipeline (round 6); GPU busy (kernel time over wall
# time, torch.profiler) 70.0% before, 94.2% with the pipelined configuration. Block lengths are illustrative.
from style import *

PREPARE, ENCODE, DECODE = 0.5, 2.1, 0.4
PERIOD = PREPARE + ENCODE + DECODE
SCALE = 1.15
X0 = -4.6
CPU_Y, GPU_Y = 0.6, -0.55
CODE = ["batch = prepare(texts)", "out   = encoder(batch)", "spans = out.tolist()", "found = decode(spans)"]
NOTES = ["# tokenize and pad", "# run the 12 layers", "# scores as numbers", "# spans and labels"]


def x(t):
    return X0 + t * SCALE


def block(y, t0, t1, color, label="", fill=0.85, height=0.6, ink=BACKGROUND):
    rect = RoundedRectangle(width=max((t1 - t0) * SCALE - 0.04, 0.04), height=height, corner_radius=0.05)
    rect.set_fill(color, fill).set_stroke(color, 1.5).move_to([(x(t0) + x(t1)) / 2, y, 0])
    if not label or rect.width < 0.9:
        return VGroup(rect)
    text = words(label, size=17, color=ink, weight=MEDIUM, font=MONO if label.startswith(".") else SANS)
    if text.width > rect.width - 0.12:
        text.scale_to_fit_width(rect.width - 0.12)
    return VGroup(rect, text.move_to(rect))


def waiting(rect, label="waiting"):
    shell = rect.copy().set_fill(BACKGROUND, 1).set_stroke(WASTE, 1.5)
    group = VGroup(shell, hatch(rect, color=WASTE))
    if label and rect.width > 0.9:
        group.add(words(label, size=17, color=INK_DIM, weight=MEDIUM).move_to(rect))
    return group


class Waiting(Clip):
    def construct(self):
        self.the_profile()
        self.two_workers()
        self.tolist_is_waiting()
        self.the_fix()
        self.finale("The slow line wasn't slow: it was where the CPU waited for the GPU.",
                    "Tesla T4 · cProfile and torch.profiler of the serving queue · rounds 5 and 6")

    # 1 ------------------------------------------------------------------------------------------------------------
    def the_profile(self):
        self.chapter(1, 4, "What the profiler said")
        lines = VGroup(*(VGroup(mono(c, size=26, color=INK), mono(n, size=22, color=INK_FAINT)) for c, n in zip(CODE, NOTES)))
        for line in lines:
            line[1].next_to(line[0], RIGHT, buff=0.5)
        lines.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
        for line in lines[1:]:
            line[1].align_to(lines[0][1], LEFT)
        back = panel(lines.width + 0.8, lines.height + 0.6).move_to(lines)
        self.code = VGroup(back, lines).move_to(UP * 1.25)
        self.say("This is the serving code for one batch of text: four lines of Python.",
                 FadeIn(back), LaggedStart(*(FadeIn(l, shift=0.1 * RIGHT) for l in lines), lag_ratio=0.25))
        self.read()

        total, left, bar_y = 10.0, -5.0, -1.3
        shares = [(".tolist()", 0.503, CPU), (".cpu()", 0.075, "#3D8DA6"), ("everything else", 0.422, INK_FAINT)]
        parts = VGroup()
        for label, share, color in shares:
            width = total * share
            rect = Rectangle(width=width - 0.03, height=0.5).set_stroke(width=0)
            rect.set_fill(color, 0.9 if color != INK_FAINT else 0.5).move_to([left + width / 2, bar_y, 0])
            text = words(label, size=18, color=BACKGROUND if color != INK_FAINT else INK, weight=MEDIUM,
                         font=MONO if label.startswith(".") else SANS)
            if text.width > width - 0.15:
                text.scale_to_fit_width(width - 0.15)
            parts.add(VGroup(rect, text.move_to(rect)))
            left += width
        title = words("cProfile of the serving thread, Tesla T4", size=19, color=INK_DIM)
        title.next_to(parts, UP, buff=0.18).align_to(parts, LEFT)
        self.say("We profiled it on the T4 to see where the time goes.",
                 FadeIn(title), LaggedStart(*(GrowFromEdge(p, LEFT) for p in parts), lag_ratio=0.2), run_time=2.4)
        self.read()

        target = lines[2][0][-9:]
        mark = box(target, CPU, fill=0.15)
        link = connector(mark, parts[0][0], color=CPU)
        share = ValueTracker(0)
        seconds = ValueTracker(0)
        number = counter(share, size=34, color=NUMBER, fmt="{:.0f}% of the time")
        number.add_updater(lambda m: m.next_to(parts, DOWN, buff=0.22).align_to(parts, LEFT))
        secs = counter(seconds, size=24, color=INK_DIM, fmt="{:.2f} s")
        secs.add_updater(lambda m: m.next_to(number, RIGHT, buff=0.35).align_to(number, DOWN))
        number.update(), secs.update()
        self.add(number, secs)
        self.say("One line took half of all the time: the call that turns the result into Python numbers.",
                 DrawBorderThenFill(mark), Create(link), share.animate.set_value(50.3), seconds.animate.set_value(2.54),
                 run_time=2.6)
        self.read()
        self.say("But it only copies a small array of scores to the CPU. That should take microseconds, not seconds.",
                 lines[0].animate.set_opacity(0.35), lines[1].animate.set_opacity(0.35), lines[3].animate.set_opacity(0.35))
        self.read(extra=0.5)
        self.profile = VGroup(title, parts)
        self.parts = parts
        self.beat(FadeOut(VGroup(mark, link, number, secs, self.profile)), *(l.animate.set_opacity(1) for l in lines),
                  run_time=1.0)
        self.remove(number, secs)

    # 2 ------------------------------------------------------------------------------------------------------------
    def two_workers(self):
        self.chapter(2, 4, "Two workers")
        lines = self.code[1]
        self.beat(self.code.animate.scale(0.62).to_corner(UR, buff=0.4), run_time=1.4)
        end = x(3 * PERIOD + 0.3)
        lanes = VGroup(*(Line([X0 - 0.15, y, 0], [end, y, 0]).set_stroke(PANEL_EDGE, 1.5) for y in (CPU_Y, GPU_Y)))
        names = VGroup(words("CPU", size=24, color=CPU, weight=MEDIUM).next_to(lanes[0], LEFT, buff=0.3),
                       words("GPU", size=24, color=GPU, weight=MEDIUM).next_to(lanes[1], LEFT, buff=0.3))
        arrow = Arrow([X0 - 0.15, GPU_Y - 0.62, 0], [end, GPU_Y - 0.62, 0], buff=0, stroke_width=2,
                      max_tip_length_to_length_ratio=0.02).set_color(INK_FAINT)
        clock = words("time", size=17, color=INK_FAINT).next_to(arrow, DOWN, buff=0.06).align_to(arrow, RIGHT)
        self.say("Two workers run this code. The CPU runs Python; the GPU runs the encoder's arithmetic.",
                 LaggedStart(*(Create(l) for l in lanes), lag_ratio=0.3), FadeIn(names, shift=0.2 * RIGHT),
                 GrowArrow(arrow), FadeIn(clock), run_time=2.2)
        self.read()
        self.timeline = VGroup(lanes, names, arrow, clock)

        pointer = Triangle(fill_color=CPU, fill_opacity=1, stroke_width=0).rotate(-PI / 2).scale(0.09)
        pointer.next_to(lines[0], LEFT, buff=0.12)
        self.pointer = pointer
        prepare = block(CPU_Y, 0, PREPARE, CPU)
        self.say("Line 1 runs on the CPU: it tokenizes the texts and pads them into one batch.",
                 FadeIn(pointer, shift=0.1 * RIGHT), GrowFromEdge(prepare, LEFT))
        self.read()

        launch = Arrow([x(PREPARE), CPU_Y - 0.3, 0], [x(PREPARE), GPU_Y + 0.3, 0], buff=0.02, stroke_width=3,
                       max_tip_length_to_length_ratio=0.3).set_color(INK)
        queued = words("queued", size=17, color=INK).next_to(launch, RIGHT, buff=0.08)
        self.say("Line 2 does not wait for the encoder. It puts the work in the GPU's queue and returns at once.",
                 pointer.animate.next_to(lines[1], LEFT, buff=0.12), GrowArrow(launch), FadeIn(queued))
        self.read()

        encode = block(GPU_Y, PREPARE, PREPARE + ENCODE, GPU, "encode batch 1")
        self.say("The GPU starts encoding, and the CPU moves straight on to line 3.",
                 pointer.animate.next_to(lines[2], LEFT, buff=0.12), GrowFromEdge(encode, LEFT), FadeOut(queued),
                 run_time=2.4)
        self.read()
        self.first = VGroup(prepare, encode, launch)

    # 3 ------------------------------------------------------------------------------------------------------------
    def tolist_is_waiting(self):
        self.chapter(3, 4, "The slow line is waiting")
        prepare, encode, launch = self.first
        lines = self.code[1]
        wait = block(CPU_Y, PREPARE, PREPARE + ENCODE, CPU, ".tolist()")
        self.say("Line 3 needs the actual numbers. They exist only once the GPU finishes, so the CPU stops here.",
                 GrowFromEdge(wait, LEFT), run_time=2.6)
        self.read()

        guides = VGroup()
        for side in (LEFT, RIGHT):
            p = wait.get_corner(DOWN + side)
            guides.add(DashedLine(p + DOWN * 0.03, [p[0], encode.get_top()[1] + 0.03, 0], dash_length=0.06)
                       .set_stroke(INK, 1.5))
        self.say("Look at the two blocks: the CPU's wait lasts exactly as long as the GPU's work.",
                 FadeOut(launch), Create(guides))
        self.read()

        self.profile.move_to(DOWN * 2.0 + RIGHT * 0.55)
        link = VGroup()
        self.say("The profiler books that wait to .tolist(). This is where its 50% came from.",
                 FadeIn(VGroup(self.profile[0], self.parts[1:])), Derive(wait, self.parts[0]), run_time=2.2)
        self.read()

        hatched = waiting(wait[0])
        bar = waiting(self.parts[0][0], "waiting for the GPU")
        self.say("So .tolist() was never slow. It was the GPU's work, seen from the CPU.",
                 ReplacementTransform(wait, hatched), ReplacementTransform(self.parts[0], bar), run_time=2.2)
        self.read(extra=0.5)

        decode = block(CPU_Y, PREPARE + ENCODE, PERIOD, CPU)
        self.say("Then the CPU decodes the spans, line 4, and starts on the next batch.",
                 FadeOut(VGroup(guides, link, self.profile, bar)),
                 self.pointer.animate.next_to(lines[3], LEFT, buff=0.12), GrowFromEdge(decode, LEFT))
        self.read()

        cpu_blocks, gpu_blocks, waits = VGroup(prepare, decode), VGroup(encode), VGroup(hatched)
        later = []
        for n in (1, 2):
            t = n * PERIOD
            later.append((block(CPU_Y, t, t + PREPARE, CPU), block(GPU_Y, t + PREPARE, t + PREPARE + ENCODE, GPU,
                                                                   f"encode batch {n + 1}"),
                          VGroup(waiting(block(CPU_Y, t + PREPARE, t + PREPARE + ENCODE, CPU)[0])),
                          block(CPU_Y, t + PREPARE + ENCODE, t + PERIOD, CPU)))
        self.say("Batch after batch, the same rhythm: prepare, launch, wait, decode.", FadeOut(self.pointer),
                 Succession(*(Succession(GrowFromEdge(p, LEFT), AnimationGroup(GrowFromEdge(e, LEFT), FadeIn(w)),
                                         GrowFromEdge(d, LEFT)) for p, e, w, d in later)), run_time=3.6)
        for p, e, w, d in later:
            cpu_blocks.add(p, d)
            gpu_blocks.add(e)
            waits.add(w)
        self.read()

        gaps = VGroup()
        for n in range(3):
            start = n * PERIOD
            for t0, t1 in [(start, start + PREPARE), (start + PREPARE + ENCODE, start + PERIOD)]:
                gap = Rectangle(width=(t1 - t0) * SCALE - 0.04, height=0.6).move_to([(x(t0) + x(t1)) / 2, GPU_Y, 0])
                gaps.add(VGroup(gap.set_stroke(INK_FAINT, 1).set_fill(opacity=0), hatch(gap, spacing=0.12, color=INK_FAINT)))
        self.say("And while the CPU decodes and prepares, the GPU has nothing queued. It sits idle.",
                 LaggedStart(*(FadeIn(g) for g in gaps), lag_ratio=0.12), run_time=2.4)
        self.read()

        self.busy = ValueTracker(0)
        self.waited = ValueTracker(0)
        busy_title = words("GPU busy", size=20, color=INK_DIM).move_to(LEFT * 4.6 + DOWN * 2.05, aligned_edge=LEFT)
        wait_title = words("CPU waiting in .tolist()", size=20, color=INK_DIM).move_to(RIGHT * 0.2 + DOWN * 2.05,
                                                                                      aligned_edge=LEFT)
        busy = counter(self.busy, size=36, color=NUMBER, fmt="{:.0f}%")
        busy.add_updater(lambda m: m.next_to(busy_title, RIGHT, buff=0.25))
        waited = counter(self.waited, size=36, color=NUMBER, fmt="{:.2f} s")
        waited.add_updater(lambda m: m.next_to(wait_title, RIGHT, buff=0.25))
        busy.update(), waited.update()
        self.add(busy, waited)
        self.say("Measured on the T4: the GPU was busy only 70% of the time, and the CPU waited 2.54 seconds.",
                 FadeIn(busy_title), FadeIn(wait_title), self.busy.animate.set_value(70),
                 self.waited.animate.set_value(2.54), run_time=2.4)
        self.read()
        self.old = VGroup(cpu_blocks, gpu_blocks, waits, gaps)

    # 4 ------------------------------------------------------------------------------------------------------------
    def the_fix(self):
        self.chapter(4, 4, "The fix: overlap")
        encode_at = [PREPARE + n * ENCODE for n in range(3)]
        before_end = DashedLine([x(3 * PERIOD), CPU_Y + 0.5, 0], [x(3 * PERIOD), GPU_Y - 0.4, 0], dash_length=0.07)
        before_end.set_stroke(INK_FAINT, 1.5)
        before = words("before", size=16, color=INK_FAINT).next_to(before_end, UP, buff=0.06)
        self.say("Keep the old finish line, and clear the timeline.", Create(before_end), FadeIn(before),
                 FadeOut(self.old, shift=0.1 * DOWN))
        self.read()

        p1 = block(CPU_Y, 0, PREPARE, CPU)
        e1 = block(GPU_Y, encode_at[0], encode_at[0] + ENCODE, GPU, "encode batch 1")
        p2 = block(CPU_Y, encode_at[0], encode_at[0] + PREPARE, CPU)
        self.say("New order: launch batch 1, then prepare batch 2 at once, while the GPU is still busy.",
                 Succession(GrowFromEdge(p1, LEFT), AnimationGroup(GrowFromEdge(e1, LEFT), GrowFromEdge(p2, LEFT))),
                 run_time=2.6)
        self.read()

        e2 = block(GPU_Y, encode_at[1], encode_at[1] + ENCODE, GPU, "encode batch 2")
        w1 = waiting(block(CPU_Y, encode_at[0] + PREPARE, encode_at[1], CPU)[0])
        self.say("Batch 2 is already queued when batch 1 finishes, so the GPU goes straight on.",
                 FadeIn(w1), GrowFromEdge(e2, LEFT), run_time=2.4)
        self.read()

        rest_cpu = VGroup(block(CPU_Y, encode_at[1], encode_at[1] + DECODE, CPU),
                          block(CPU_Y, encode_at[1] + DECODE, encode_at[1] + DECODE + PREPARE, CPU),
                          block(CPU_Y, encode_at[2], encode_at[2] + DECODE, CPU),
                          block(CPU_Y, encode_at[2] + ENCODE, encode_at[2] + ENCODE + DECODE, CPU))
        e3 = block(GPU_Y, encode_at[2], encode_at[2] + ENCODE, GPU, "encode batch 3")
        rest_wait = VGroup(*(waiting(block(CPU_Y, t0, t1, CPU)[0], label="")
                             for t0, t1 in [(encode_at[1] + DECODE + PREPARE, encode_at[2]),
                                            (encode_at[2] + DECODE, encode_at[2] + ENCODE)]))
        self.say("The CPU decodes one batch while the GPU encodes the next. Its waits shrink to the leftovers.",
                 LaggedStart(GrowFromEdge(rest_cpu[0], LEFT), GrowFromEdge(rest_cpu[1], LEFT), GrowFromEdge(e3, LEFT),
                             FadeIn(rest_wait[0]), GrowFromEdge(rest_cpu[2], LEFT), FadeIn(rest_wait[1]),
                             GrowFromEdge(rest_cpu[3], LEFT), lag_ratio=0.35), run_time=3.4)
        self.read()

        finish = encode_at[2] + ENCODE + DECODE
        now_end = DashedLine([x(finish), CPU_Y + 0.5, 0], [x(finish), GPU_Y - 0.4, 0], dash_length=0.07)
        now_end.set_stroke(CORRECT, 2)
        now = words("now", size=16, color=CORRECT).next_to(now_end, UP, buff=0.06)
        span_arrow = DoubleArrow([x(finish), CPU_Y + 0.95, 0], [x(3 * PERIOD), CPU_Y + 0.95, 0], buff=0, stroke_width=2,
                                 tip_length=0.12).set_color(CORRECT)
        self.say("The GPU's gaps close: the three batches run back to back and finish earlier.",
                 Create(now_end), FadeIn(now), GrowFromCenter(span_arrow), run_time=2.2)
        self.read()

        self.say("Measured on the T4, the GPU is now busy 94% of the time.", self.busy.animate.set_value(94.2),
                 run_time=2.4)
        self.read()
        self.say("And the time booked to .tolist() fell from 2.54 to 0.78 seconds. Same code, new order.",
                 self.waited.animate.set_value(0.78), run_time=2.4)
        self.read(extra=0.5)

In [ ]:
video = render("clip_b_waiting.py", "Waiting")
gif(video, "clip_b_waiting.gif")

## C. Overlap keeps entities whole

> Overlap means no entity is ever only seen cut in half.

In [ ]:
%%writefile clip_c_overlap.py
# Clip C: "Overlap means no entity is ever only seen cut in half."
# Chapters: 1 the model reads a long document one window at a time; 2 fixed chunks cut a name in half and each half is
# judged alone; 3 overlapping windows repeat the last 64 words, so one window holds the whole name; 4 the merge keeps one
# span, and the measured result and cost.
# Numbers: realistic corpus (040 sections), round 9 of benchmark_results/pipeline_2026-10-25: 51 annotated names cut by
# pre-cut chunk boundaries, 0 with GLiNER2's 384-word windows overlapping by 64; windowing costs +22% to +27% time.
from style import *

FILLER = [
    "This services agreement is made between the customer and the supplier named below.",
    "The supplier will provide the services described in the approved schedule of work.",
    "Each deliverable is reviewed against the acceptance criteria in that schedule.",
    "Fees follow the rates and quantities in the approved order for each period.",
    "Invoices identify the service period, the order reference and any variations.",
    "Disputed lines are held while the parties reconcile the supporting evidence.",
    "Both parties keep the records needed to reconstruct an approval decision.",
    "Personnel with the right experience are assigned to the scheduled activities.",
    "Access to customer systems is limited to the approved working purpose only.",
    "Information supplied during delivery is used only for the agreed purpose.",
    "During the term, notices for the customer go to Lakshmi Varma at the Kochi office.",
    "A proposed change describes the affected deliverable and its revised window.",
    "Dependencies that threaten the plan are recorded promptly by either party.",
    "The representatives review each variation before it is authorised in writing.",
    "Escalations name the service reference and the response that is requested.",
    "The receiving party acknowledges each notice and assigns an owner to it.",
    "Operational questions are first reviewed by the delivery coordinators.",
    "Unresolved questions go to the authorised representatives with evidence.",
]
NAME_LINE = 10
SOURCE = "…notices for the customer go to Lakshmi Varma at the Kochi office…"


class Overlap(Clip):
    def construct(self):
        self.reading_in_windows()
        self.cutting_without_overlap()
        self.overlapping_windows()
        self.merge_and_result()
        self.finale("Overlap means no entity is ever only seen cut in half.",
                    "realistic test corpus · GLiNER2 windows of 384 words, 64 overlap")

    def reading_in_windows(self):
        self.chapter(1, 4, "A long document, read one window at a time")
        page = panel(9.2, 4.75).move_to(UP * 0.45)
        lines = VGroup(*(words(t, size=12, color=INK_DIM) for t in FILLER)).arrange(DOWN, aligned_edge=LEFT, buff=0.11)
        lines.move_to(page)
        pitch = lines[0].get_center()[1] - lines[1].get_center()[1]
        self.page, self.lines = page, lines
        self.say("A contract can run to 50,000 words. Here is a small piece of one.",
                 FadeIn(page), LaggedStart(*(FadeIn(l, shift=0.05 * UP) for l in lines), lag_ratio=0.04))
        self.read()

        window = Rectangle(width=page.width - 0.3, height=6 * pitch).set_stroke(GPU, 3).set_fill(GPU, 0.07)
        window.move_to(lines[0:6]).align_to(lines[0], UP).shift(UP * 0.06)
        tag = words("one window", size=18, color=GPU).next_to(window, RIGHT, buff=0.15).align_to(window, UP)
        self.say("GLiNER2 reads at most about 384 words at a time: one window.",
                 Create(window), FadeIn(tag, shift=0.1 * LEFT))
        self.read()
        self.say("So a long document is read window by window, and each window is judged on its own.",
                 VGroup(window, tag).animate.shift(DOWN * 6 * pitch), run_time=2.4)
        self.beat(VGroup(window, tag).animate.shift(DOWN * 6 * pitch), run_time=1.6)
        self.read()
        self.beat(FadeOut(window), FadeOut(tag), run_time=0.8)

    def cutting_without_overlap(self):
        self.chapter(2, 4, "Pre-cut chunks: a fixed word count")
        line = words(SOURCE, size=30, color=INK_DIM).move_to(UP * 1.55)
        self.line = line
        source_line = self.lines[NAME_LINE]
        self.say("Look closely at one line of it.",
                 self.lines.animate.set_opacity(0.15), source_line.animate.set_color(INK).set_opacity(1))
        self.beat(Derive(source_line, line), FadeOut(self.page), FadeOut(self.lines), run_time=1.8)
        first, last = span(line, SOURCE, "Lakshmi"), span(line, SOURCE, "Varma")
        self.first, self.last = first, last
        self.beat(VGroup(first, last).animate.set_color(INK), run_time=0.8)
        self.read()

        self.cut_x = cut_x = (first.get_right()[0] + last.get_left()[0]) / 2
        left_edge, right_edge = line.get_left()[0] - 0.1, line.get_right()[0] + 0.1
        self.edges = (left_edge, right_edge)
        y = line.get_bottom()[1] - 0.3
        chunks = VGroup(bracket(left_edge, cut_x - 0.05, y, up=False), bracket(cut_x + 0.05, right_edge, y, up=False))
        names = VGroup(words("chunk 1", size=19, color=INK_DIM).next_to(chunks[0], DOWN, buff=0.1),
                       words("chunk 2", size=19, color=INK_DIM).next_to(chunks[1], DOWN, buff=0.1))
        cut = DashedLine([cut_x, line.get_top()[1] + 0.3, 0], [cut_x, y - 0.1, 0], dash_length=0.08).set_stroke(ERROR, 3)
        self.say("The old pipeline cut the text into chunks every N words, wherever that landed.",
                 Create(chunks[0]), Create(chunks[1]), FadeIn(names))
        self.read()
        self.say("Here the cut falls between “Lakshmi” and “Varma”.", Create(cut), run_time=1.4)
        self.read()

        views = VGroup()
        for title, text in [("chunk 1 sees", "…for the customer go to Lakshmi"), ("chunk 2 sees", "Varma at the Kochi office…")]:
            body = words(text, size=22, color=INK_DIM)
            frame = panel(body.width + 0.7, 1.15)
            head = words(title, size=17, color=INK_DIM).next_to(frame, UP, buff=0.1).align_to(frame, LEFT)
            views.add(VGroup(frame, body.move_to(frame), head))
        views.arrange(RIGHT, buff=0.6).move_to(DOWN * 0.3)
        sources = (span(line, SOURCE, "for the customer go to Lakshmi"), span(line, SOURCE, "Varma at the Kochi office"))
        self.say("The model never sees the two halves together. Each chunk is judged alone.",
                 *(Derive(part, view[1]) for part, view in zip(sources, views)),
                 *(FadeIn(VGroup(v[0], v[2])) for v in views), run_time=2.2)
        self.read()
        lakshmi = span(views[0][1], "…for the customer go to Lakshmi", "Lakshmi")
        varma = span(views[1][1], "Varma at the Kochi office…", "Varma")
        half = box(lakshmi, ERROR, fill=0.16)
        missed = DashedVMobject(SurroundingRectangle(varma, buff=0.07, corner_radius=0.06), num_dashes=24)
        missed.set_stroke(ERROR, 2)
        notes = VGroup(words("“Lakshmi”: half a name", size=18, color=ERROR).next_to(views[0], DOWN, buff=0.15),
                       words("“Varma” alone: missed", size=18, color=ERROR).next_to(views[1], DOWN, buff=0.15))
        self.say("Chunk 1 reports half a name. Chunk 2 sees “Varma” without its first name and misses it.",
                 DrawBorderThenFill(half), Create(missed), FadeIn(notes, shift=0.1 * UP), run_time=2.2)
        self.read()

        self.tally = ValueTracker(0)
        self.tally_color = ERROR
        number = counter(self.tally, size=54, color=lambda: self.tally_color)
        label = words("names cut in half", size=22, color=INK_DIM)
        number.add_updater(lambda m: m.move_to(DOWN * 1.6 + LEFT * 1.4))
        label.add_updater(lambda m: m.next_to(number, RIGHT, buff=0.25).align_to(number, DOWN))
        number.update()
        label.update()
        self.number, self.number_label = number, label
        self.beat(FadeOut(views), FadeOut(half), FadeOut(missed), FadeOut(notes), run_time=0.8)
        self.add(number)
        self.say("In our ten test documents, 51 names were cut like this.",
                 FadeIn(label), self.tally.animate.set_value(51), run_time=2.4)
        self.read()
        self.chunk_marks = VGroup(chunks, names, cut)

    def overlapping_windows(self):
        self.chapter(3, 4, "Overlapping windows")
        line, (left_edge, right_edge) = self.line, self.edges
        top = line.get_top()[1]
        w1_right = self.cut_x - 0.05
        w2_left = span(line, SOURCE, "customer").get_left()[0] - 0.1
        window_1 = bracket(left_edge, w1_right, top + 0.45)
        window_2 = bracket(w2_left, right_edge, top + 0.95)
        heads = VGroup(words("window 1", size=19, color=INK_DIM).next_to(window_1, UP, buff=0.08).align_to(window_1, LEFT),
                       words("window 2", size=19, color=INK_DIM).next_to(window_2, UP, buff=0.08).align_to(window_2, RIGHT))
        self.say("GLiNER2's own long-document mode does it differently: its windows overlap.",
                 FadeOut(self.chunk_marks), Create(window_1), FadeIn(heads[0]))
        self.read()
        overlap = Rectangle(width=w1_right - w2_left, height=line.height + 0.3).set_stroke(width=0)
        overlap.set_fill(GPU, 0.14).move_to([(w1_right + w2_left) / 2, line.get_center()[1], 0])
        overlap_tag = words("read twice", size=18, color=GPU).next_to(overlap, DOWN, buff=0.1)
        self.say("Each window starts 64 words before the previous one ends, so those words are read twice.",
                 Create(window_2), FadeIn(heads[1]), FadeIn(overlap), FadeIn(overlap_tag), run_time=2.4)
        self.read()

        text = "customer go to Lakshmi Varma at the Kochi office…"
        body = words(text, size=22, color=INK_DIM)
        frame = panel(body.width + 0.7, 1.15)
        head = words("window 2 sees", size=17, color=INK_DIM).next_to(frame, UP, buff=0.1).align_to(frame, LEFT)
        view = VGroup(frame, body.move_to(frame), head).move_to(DOWN * 0.05)
        whole = span(body, text, "Lakshmi Varma")
        found = box(whole, CORRECT, fill=0.2)
        self.say("Window 2 starts early enough to hold the whole name, so it finds “Lakshmi Varma”.",
                 Derive(span(line, SOURCE, "customer go to Lakshmi Varma at the Kochi office"), body),
                 FadeIn(VGroup(frame, head)), run_time=2.2)
        self.beat(DrawBorderThenFill(found), whole.animate.set_color(INK), run_time=1.2)
        self.read()
        self.say("The rule: a name shorter than the 64-word overlap always fits whole inside some window.",
                 overlap.animate.set_fill(GPU, 0.28), run_time=1.4)
        self.read()
        self.window_parts = VGroup(window_1, window_2, heads, overlap, overlap_tag)
        self.view = VGroup(view, found)

    def merge_and_result(self):
        self.chapter(4, 4, "Merging, and the measured result")
        partial = box(self.first, WASTE, fill=0.12)
        full = box(VGroup(self.first, self.last), CORRECT, fill=0.2)
        tags = VGroup(words("from window 1", size=17, color=WASTE).next_to(partial, DOWN, buff=0.55),
                      words("from window 2", size=17, color=CORRECT).next_to(full, DOWN, buff=0.55))
        tags[0].shift(LEFT * 0.8)
        tags[1].shift(RIGHT * 0.9)
        self.say("Both windows report what they saw, mapped back onto the document.",
                 FadeOut(self.view), self.window_parts.animate.set_opacity(0.3),
                 DrawBorderThenFill(partial), DrawBorderThenFill(full), FadeIn(tags))
        self.read()
        self.say("Where reports overlap, the merge keeps one span: here, the whole name.",
                 FadeOut(partial, scale=0.9), FadeOut(tags[0]), run_time=1.6)
        self.read()
        self.say("Across the same ten documents, the names cut in half dropped from 51 to 0.",
                 self.tally.animate.set_value(0), run_time=2.4)
        self.tally_color = CORRECT
        self.beat(Flash(self.number, color=CORRECT, flash_radius=0.6, line_length=0.25), run_time=0.8)
        self.read()
        cost = words("cost: 64 of every 384 words are read twice, about 22% more time", size=20, color=INK_DIM)
        cost.next_to(VGroup(self.number, self.number_label), DOWN, buff=0.2)
        self.say("The price is re-reading the overlap: about a fifth more time for the whole document.",
                 FadeIn(cost, shift=0.1 * UP))
        self.read(extra=0.5)

In [ ]:
video = render("clip_c_overlap.py", "Overlap")
gif(video, "clip_c_overlap.gif")

## D. DeBERTa's three attentions

> DeBERTa looks at what words are and where they are, so it needs a special kernel.

In [ ]:
%%writefile clip_d_deberta.py
# Clip D: "DeBERTa looks at what words are and where they are, so it needs a special kernel."
# Chapters: 1 ordinary attention (one worked pair: two word vectors, one dot product, one cell; then the whole grid);
# 2 where words are (the grid ignores order; DeBERTa measures the distance between two words, constant along each
# diagonal); 3 three questions (content-content, content-position, position-content, each a grid; they add up to the
# score); 4 a special kernel (fast generic kernels build only the first grid; eager PyTorch writes all three to memory;
# a fused kernel computes all three tile by tile).
# Numbers: Tesla T4, mDeBERTa-v3 encoder, batch 16 x 512 tokens: eager fp16 295 ms, Triton disentangled_flash 143 ms,
# glinert4 CUDA kernel 113 ms (kernel log rows 22 and 40). The grid values are illustrative.
from style import *

TOKENS = ["Lakshmi", "Varma", "signed", "the", "contract"]
N = len(TOKENS)
C2C = [[1.0, .85, .25, .05, .2], [.85, 1.0, .3, .05, .25], [.3, .35, 1.0, .2, .8], [.05, .05, .25, 1.0, .45],
       [.2, .25, .85, .4, 1.0]]
VECTORS = {"Lakshmi": [.9, .2, .7, .1, .8, .3], "Varma": [.8, .3, .6, .2, .9, .2]}


def by_distance(weights, flip=False):
    return [[weights[(i - j if flip else j - i) + N - 1] for j in range(N)] for i in range(N)]


C2P = by_distance([.05, .1, .2, .7, .5, .95, .35, .15, .05])
P2C = by_distance([.05, .15, .3, .6, .45, .9, .4, .1, .05], flip=True)
TOTAL = [[min(1, (a + b + c) / 2.2) for a, b, c in zip(r1, r2, r3)] for r1, r2, r3 in zip(C2C, C2P, P2C)]


def vector(values, color, cell=0.34):
    return VGroup(*(Square(cell).set_stroke(color, 1).set_fill(color, 0.1 + 0.85 * v) for v in values)).arrange(
        DOWN, buff=0.04)


def cell(grid, i, j):
    return grid[i * N + j]


class ThreeAttentions(Clip):
    def construct(self):
        self.ordinary()
        self.where_words_are()
        self.three_questions()
        self.special_kernel()
        self.finale("DeBERTa looks at what words are and where they are, so it needs a special kernel.",
                    "Tesla T4 · mDeBERTa-v3 encoder · batch 16 × 512 tokens", size=26)

    # 1 ------------------------------------------------------------------------------------------------------------
    def ordinary(self):
        self.chapter(1, 4, "Ordinary attention")
        sentence = words(" ".join(TOKENS), size=36).move_to(UP * 1.6)
        self.say("A sentence. To understand it, each word looks at every other word.",
                 LaggedStart(*(FadeIn(ch, shift=0.1 * UP) for ch in sentence), lag_ratio=0.02))
        self.read()

        grid = heat_grid(C2C, TERM_C2C, cell=0.5, gap=0.05).move_to(LEFT * 2.6 + DOWN * 0.35)
        for sq in grid:
            sq.set_fill(TERM_C2C, 0.04).set_stroke(PANEL_EDGE, 1)
        cols = VGroup(*(words(t, size=18, color=INK_DIM) for t in TOKENS))
        rows = VGroup(*(words(t, size=18, color=INK_DIM) for t in TOKENS))
        for k, (c, r) in enumerate(zip(cols, rows)):
            c.rotate(PI / 4).move_to(cell(grid, 0, k).get_top() + UP * 0.08 + LEFT * 0.1, aligned_edge=DL)
            r.next_to(cell(grid, k, 0), LEFT, buff=0.2)
        pieces = [span(sentence, " ".join(TOKENS), t) for t in TOKENS]
        self.say("Lay the words out twice, as rows and as columns. Every pair of words gets one cell.",
                 FadeOut(sentence), *(Derive(p, c) for p, c in zip(pieces, cols)),
                 *(Derive(p.copy(), r) for p, r in zip(pieces, rows)), FadeIn(grid), run_time=2.4)
        self.read()

        target = cell(grid, 0, 1)
        focus = SurroundingRectangle(target, buff=0.03, corner_radius=0.04).set_stroke(INK, 2.5)
        dim = [sq.animate.set_stroke(opacity=0.35) for sq in grid if sq is not target]
        self.say("Take one cell: the row word Lakshmi, looking at the column word Varma.",
                 Create(focus), rows[0].animate.set_color(INK), cols[1].animate.set_color(INK),
                 *(r.animate.set_opacity(0.4) for r in rows[1:]), *(c.animate.set_opacity(0.4) for c in cols if c is not cols[1]),
                 *dim)
        self.read()

        left = vector(VECTORS["Lakshmi"], TERM_C2C)
        right = vector(VECTORS["Varma"], TERM_C2C)
        dot = words("·", size=48, color=INK_DIM)
        eq = words("=", size=36, color=INK_DIM)
        score = ValueTracker(0)
        value = counter(score, size=40, color=NUMBER, fmt="{:.2f}")
        example = VGroup(left, dot, right, eq).arrange(RIGHT, buff=0.35).move_to(RIGHT * 3.3 + DOWN * 0.35)
        value.add_updater(lambda m: m.next_to(eq, RIGHT, buff=0.3))
        value.update()
        names = VGroup(words("Lakshmi", size=18, color=INK).next_to(left, UP, buff=0.15),
                       words("Varma", size=18, color=INK).next_to(right, UP, buff=0.15))
        self.say("Each word is stored as a vector: a list of numbers describing what it means.",
                 Derive(rows[0].copy(), names[0]), Derive(cols[1].copy(), names[1]),
                 LaggedStart(GrowFromEdge(left, UP), GrowFromEdge(right, UP), lag_ratio=0.3))
        self.read()
        self.add(value)
        self.say("Multiply the two vectors entry by entry and add up: a high score means the words belong together.",
                 FadeIn(dot), FadeIn(eq), score.animate.set_value(0.85), run_time=2.4)
        self.read()
        link = connector(target, VGroup(left, right), color=INK_DIM)
        self.say("That score colours the cell. Bright means strongly related.",
                 target.animate.set_fill(TERM_C2C, 0.08 + 0.85 * 0.85))
        self.read()

        self.say("Do this for every pair and the whole grid fills in. That is ordinary attention.",
                 FadeOut(VGroup(example, names, focus, link)), FadeOut(value),
                 *(r.animate.set_opacity(1).set_color(INK_DIM) for r in rows),
                 *(c.animate.set_opacity(1).set_color(INK_DIM) for c in cols),
                 LaggedStart(*(sq.animate.set_fill(TERM_C2C, 0.08 + 0.85 * C2C[k // N][k % N]).set_stroke(width=0)
                               for k, sq in enumerate(grid)), lag_ratio=0.03), run_time=2.6)
        self.read()
        self.grid, self.cols, self.rows = grid, cols, rows

    # 2 ------------------------------------------------------------------------------------------------------------
    def where_words_are(self):
        self.chapter(2, 4, "Where words are")
        grid = self.grid
        swapped = VGroup(words("Lakshmi Varma", size=24, color=INK), words("Varma Lakshmi", size=24, color=INK))
        swapped.arrange(DOWN, buff=0.3).move_to(RIGHT * 3.6 + UP * 0.6)
        same = words("same scores", size=20, color=ERROR).next_to(swapped, DOWN, buff=0.3)
        self.say("But the score only depends on what the two words are, not on where they are.",
                 FadeIn(swapped[0], shift=0.1 * UP))
        self.read()
        self.say("Swap the words around, and ordinary attention sees exactly the same pairs.",
                 Derive(swapped[0].copy(), swapped[1]), FadeIn(same, shift=0.1 * UP))
        self.read()

        numbers = VGroup()
        for i in range(N):
            for j in range(N):
                numbers.add(words(f"{j - i:+d}" if j != i else "0", size=18, color=TERM_C2P, font=MONO)
                            .move_to(cell(grid, i, j)))
        board = VGroup(*(sq.copy().set_fill(PANEL, 1).set_stroke(PANEL_EDGE, 1) for sq in grid))
        self.say("DeBERTa also measures where: how far the column word is from the row word.",
                 FadeOut(VGroup(swapped, same)), FadeIn(board), LaggedStart(*(FadeIn(n) for n in numbers), lag_ratio=0.03),
                 run_time=2.6)
        self.read()

        diagonal = VGroup(*(cell(grid, i, i + 1) for i in range(N - 1)))
        stripe = VGroup(*(SurroundingRectangle(sq, buff=0.02, corner_radius=0.03).set_stroke(INK, 2) for sq in diagonal))
        note = words("+1: the next word", size=20, color=TERM_C2P).move_to(RIGHT * 3.6 + UP * 0.6)
        self.say("Every cell on this diagonal holds the same distance: the next word along.",
                 LaggedStart(*(Create(r) for r in stripe), lag_ratio=0.15), FadeIn(note, shift=0.1 * UP))
        self.read()
        note2 = words("−1: the word before", size=20, color=TERM_C2P).next_to(note, DOWN, buff=0.2).align_to(note, LEFT)
        back = cell(grid, 1, 0)
        mark = SurroundingRectangle(back, buff=0.02, corner_radius=0.03).set_stroke(INK, 2)
        self.say("And now order matters: Varma is +1 from Lakshmi, Lakshmi is −1 from Varma.",
                 Create(mark), FadeIn(note2, shift=0.1 * UP))
        self.read()
        self.say("Each distance has its own learned vector, just like each word.",
                 FadeOut(VGroup(stripe, mark, note, note2)))
        self.read()
        self.board, self.numbers = board, numbers

    # 3 ------------------------------------------------------------------------------------------------------------
    def three_questions(self):
        self.chapter(3, 4, "Three questions")
        grids = VGroup(heat_grid(C2C, TERM_C2C), heat_grid(C2P, TERM_C2P), heat_grid(P2C, TERM_P2C), heat_grid(TOTAL, INK))
        plus1, plus2, equals = (words(s, size=40, color=INK_DIM) for s in ("+", "+", "="))
        row = VGroup(grids[0], plus1, grids[1], plus2, grids[2], equals, grids[3]).arrange(RIGHT, buff=0.42)
        row.move_to(UP * 0.55)
        heads = [("what ↔ what", "content to content", TERM_C2C), ("what → where", "content to position", TERM_C2P),
                 ("where → what", "position to content", TERM_P2C), ("the score", "summed, then softmax", INK)]
        captions = VGroup()
        for grid, (head, sub, color) in zip(grids, heads):
            h = words(head, size=21, color=color, weight=MEDIUM).next_to(grid, DOWN, buff=0.25)
            captions.add(VGroup(h, words(sub, size=16, color=INK_DIM).next_to(h, DOWN, buff=0.08)))

        distances = VGroup(self.board, self.numbers)
        ghosts = [distances.copy().scale(g.width / self.board.width).move_to(g) for g in grids[1:3]]
        self.say("Question one is the grid we already have: how related are the two words' meanings?",
                 FadeOut(distances), FadeOut(VGroup(self.cols, self.rows)), ReplacementTransform(self.grid, grids[0]),
                 FadeIn(captions[0], shift=0.1 * UP), run_time=2.4)
        self.read()

        ghost = ghosts[0]
        self.say("Question two: how does this word relate to that distance? Lakshmi asks: what usually sits one word after me?",
                 FadeIn(plus1), FadeIn(ghost), run_time=1.6)
        self.beat(FadeOut(ghost[1]), ReplacementTransform(ghost[0], grids[1]), FadeIn(captions[1], shift=0.1 * UP),
                  run_time=2.0)
        self.read()
        diagonal = VGroup(*(cell(grids[1], i, i + 1) for i in range(N - 1)))
        stripe = VGroup(*(SurroundingRectangle(sq, buff=0.02, corner_radius=0.03).set_stroke(INK, 2) for sq in diagonal))
        self.say("Because it is built from distances, its pattern runs along the diagonals.",
                 LaggedStart(*(Create(r) for r in stripe), lag_ratio=0.15))
        self.read()

        ghost = ghosts[1]
        self.say("Question three is the mirror image: how does that distance relate to the other word?",
                 FadeOut(stripe), FadeIn(plus2), FadeIn(ghost), run_time=1.6)
        self.beat(FadeOut(ghost[1]), ReplacementTransform(ghost[0], grids[2]), FadeIn(captions[2], shift=0.1 * UP),
                  run_time=2.0)
        self.read()

        sources = VGroup(grids[0].copy(), grids[1].copy(), grids[2].copy())
        self.say("Add the three grids cell by cell. That sum is the attention score DeBERTa uses.",
                 FadeIn(equals), *(Transform(src, grids[3]) for src in sources), FadeIn(captions[3], shift=0.1 * UP),
                 run_time=2.6)
        self.remove(*sources)
        self.add(grids[3])
        self.read(extra=0.5)
        self.row, self.grids, self.captions = row, grids, captions

    # 4 ------------------------------------------------------------------------------------------------------------
    def special_kernel(self):
        self.chapter(4, 4, "Why a special kernel")
        grids = self.grids
        crosses = VGroup(*(VGroup(Line(g.get_corner(UL), g.get_corner(DR)), Line(g.get_corner(UR), g.get_corner(DL)))
                           .set_stroke(ERROR, 4) for g in grids[1:3]))
        self.say("Fast generic attention kernels, like FlashAttention, only know how to build the first grid.",
                 LaggedStart(*(Create(c) for c in crosses), lag_ratio=0.3))
        self.read()
        fused = SurroundingRectangle(VGroup(grids[0], grids[2]), buff=0.2, corner_radius=0.12).set_stroke(GPU, 3)
        self.say("So DeBERTa needs its own kernel: one that computes all three, piece by piece, without writing them out.",
                 FadeOut(crosses), Create(fused), run_time=2.2)
        self.read()

        self.beat(FadeOut(self.captions), VGroup(self.row, fused).animate.scale(0.62).to_edge(UP, buff=1.2), run_time=1.4)
        chart_title = words("whole encoder, batch 16 × 512 tokens, Tesla T4", size=20, color=INK_DIM)
        bars = VGroup()
        unit = 6.0 / 295
        rows = [("eager PyTorch fp16", 295, WASTE), ("Triton disentangled_flash", 143, INK_DIM),
                ("glinert4 CUDA kernel", 113, GPU)]
        for label, value, color in rows:
            name = words(label, size=20, color=INK)
            bar = Rectangle(width=value * unit, height=0.36).set_fill(color, 0.9).set_stroke(width=0)
            number = words(f"{value} ms", size=20, color=NUMBER if color == GPU else INK, weight=BOLD, font=MONO)
            bars.add(VGroup(name, bar, number))
        bar_left, top = -1.4, -0.45
        for k, item in enumerate(bars):
            y = top - k * 0.62
            item[1].move_to([bar_left, y, 0], aligned_edge=LEFT)
            item[0].move_to([bar_left - 0.3, y, 0], aligned_edge=RIGHT)
            item[2].next_to(item[1], RIGHT, buff=0.2)
        chart_title.next_to(bars, UP, buff=0.3).align_to(bars, LEFT)
        texts = ["Plain PyTorch writes all three grids to GPU memory and reads them back: 295 ms for the encoder.",
                 "A Triton kernel that fuses the three terms halves that: 143 ms.",
                 "Our CUDA kernel, written for the T4's tensor cores: 113 ms."]
        for k, item in enumerate(bars):
            extra = [FadeIn(chart_title)] if k == 0 else []
            self.say(texts[k], *extra, FadeIn(item[0]), GrowFromEdge(item[1], LEFT), FadeIn(item[2]), run_time=2.0)
            self.read()

In [ ]:
video = render("clip_d_deberta.py", "ThreeAttentions")
gif(video, "clip_d_deberta.gif")

## E. Feeding the tensor cores

> The kernel re-packs the work into the exact shape the T4's fast units want.

In [ ]:
%%writefile clip_e_tiles.py
# Clip E: "The kernel re-packs the work into the exact shape the T4's fast units want."
# Chapters: 1 a tensor core (one worked output cell: a row times a column; one instruction does the whole 16x8 tile);
# 2 where the data lives (GPU memory: large and far; on-chip memory: small and fast; tensor cores need tiles close by);
# 3 before (the Triton kernel read Q, K and the position tables in scattered pieces, so the tensor cores idled);
# 4 now (one wide read per tile into on-chip memory, only the distances a tile needs, the next tile prefetched into
# registers by hand because the T4 has no asynchronous copy; tiles stream into the tensor cores).
# Numbers: Tesla T4, 16 x 12 heads x 512 tokens, per layer (kernel log rows 24, 25, 31, 38): Triton kernel 4.80 ms +
# position tables 2.2 ms = 7.0 ms; ours 2.84 ms; Triton disentangled kernel 2.7 TFLOP/s vs plain attention (SDPA)
# 17.0 TFLOP/s. T4 spec: 65 TFLOP/s fp16 tensor-core peak, 320 GB/s memory, 64 KB on-chip memory per SM.
from style import *

A_VALUES = [[((r * 5 + c * 3) % 7) / 6 for c in range(8)] for r in range(16)]
B_VALUES = [[((r * 3 + c * 5) % 7) / 6 for c in range(8)] for r in range(8)]


def tile(rows, cols, color, cell=0.11, gap=0.018, fill=0.85, values=None):
    grid = VGroup(*(Square(cell).set_stroke(width=0).set_fill(color, fill if values is None else
                                                              0.25 + 0.7 * values[k // cols][k % cols])
                    for k in range(rows * cols)))
    return grid.arrange_in_grid(rows=rows, cols=cols, buff=gap)


def area(width, height, title, note, edge=PANEL_EDGE):
    box = panel(width, height, edge=edge)
    head = words(title, size=21, color=INK, weight=MEDIUM).next_to(box, UP, buff=0.14).align_to(box, LEFT)
    sub = words(note, size=16, color=INK_DIM).next_to(head, RIGHT, buff=0.2).align_to(head, DOWN)
    return VGroup(box, head, sub)


class Tiles(Clip):
    def construct(self):
        self.tensor_core()
        self.where_data_lives()
        self.before()
        self.now()
        self.finale("The kernel re-packs the work into the exact shape the T4's fast units want.",
                    "Tesla T4 · attention per layer, 16 × 12 heads × 512 tokens", size=28)

    # 1 ------------------------------------------------------------------------------------------------------------
    def tensor_core(self):
        self.chapter(1, 4, "What a tensor core does")
        cell = 0.17
        a = tile(16, 8, CPU, cell=cell, gap=0.025, values=A_VALUES)
        b = tile(8, 8, TERM_C2P, cell=cell, gap=0.025, values=B_VALUES)
        c = tile(16, 8, GPU, cell=cell, gap=0.025, fill=0.0)
        for sq in c:
            sq.set_stroke(PANEL_EDGE, 1)
        times, equals = words("×", size=40, color=INK_DIM), words("=", size=40, color=INK_DIM)
        product = VGroup(a, times, b, equals, c).arrange(RIGHT, buff=0.45).move_to(UP * 0.35)
        sizes = VGroup(*(words(n, size=18, color=INK_DIM, font=MONO).next_to(g, DOWN, buff=0.15)
                         for g, n in ((a, "16 × 8"), (b, "8 × 8"), (c, "16 × 8"))))
        self.say("Attention is mostly multiplying blocks of numbers. Here is one block: 16 rows of 8 numbers.",
                 LaggedStart(*(FadeIn(sq, scale=0.5) for sq in a), lag_ratio=0.01), FadeIn(sizes[0]))
        self.read()
        self.say("And a second block, 8 by 8, to multiply it with.",
                 FadeIn(times), LaggedStart(*(FadeIn(sq, scale=0.5) for sq in b), lag_ratio=0.01), FadeIn(sizes[1]))
        self.read()

        row = VGroup(*a[16:24])
        col = VGroup(*(b[k * 8 + 3] for k in range(8)))
        out = c[2 * 8 + 3]
        row_box = SurroundingRectangle(row, buff=0.03, corner_radius=0.03).set_stroke(INK, 2)
        col_box = SurroundingRectangle(col, buff=0.03, corner_radius=0.03).set_stroke(INK, 2)
        self.say("One cell of the answer takes one row of the first block and one column of the second.",
                 FadeIn(equals), FadeIn(c), FadeIn(sizes[2]), Create(row_box), Create(col_box),
                 *(sq.animate.set_opacity(0.3) for sq in a if sq not in row),
                 *(sq.animate.set_opacity(0.3) for sq in b if sq not in col))
        self.read()
        pairs = VGroup(*(Line(r.get_center(), k.get_center()).set_stroke(INK, 1.2, opacity=0.6) for r, k in zip(row, col)))
        self.say("Eight multiplications, added up, give one number.",
                 LaggedStart(*(ShowPassingFlash(p.copy().set_stroke(NUMBER, 3, 1), time_width=0.5) for p in pairs),
                             lag_ratio=0.12),
                 out.animate.set_fill(GPU, 0.85).set_stroke(GPU, 2), run_time=2.4)
        self.read()

        instruction = words("mma.sync m16n8k8: fp16 in, fp32 sums", size=20, color=GPU, font=MONO)
        instruction.next_to(sizes, DOWN, buff=0.35)
        self.say("A tensor core does that for all 128 cells at once, in a single instruction: 1,024 multiplications.",
                 FadeOut(row_box), FadeOut(col_box), *(sq.animate.set_opacity(1) for sq in [*a, *b]),
                 LaggedStart(*(sq.animate.set_fill(GPU, 0.85).set_stroke(width=0) for sq in c), lag_ratio=0.01),
                 FadeIn(instruction, shift=0.1 * UP), run_time=2.6)
        self.read()
        self.say("Kept fed, the T4's tensor cores can do 65 trillion of these a second. The hard part is keeping them fed.")
        self.read(extra=0.5)
        self.product = VGroup(product, sizes, instruction)

    # 2 ------------------------------------------------------------------------------------------------------------
    def where_data_lives(self):
        self.chapter(2, 4, "Where the data lives")
        memory = area(3.0, 3.2, "GPU memory", "large, far away").move_to(LEFT * 4.9 + DOWN * 0.1)
        chip_mem = area(3.0, 3.2, "on-chip memory", "64 KB, fast").move_to(LEFT * 0.4 + DOWN * 0.1)
        core_box = RoundedRectangle(width=2.6, height=1.6, corner_radius=0.16).set_fill(PANEL, 1).set_stroke(GPU, 2.5)
        core_box.move_to(RIGHT * 4.5 + DOWN * 0.1)
        core = VGroup(core_box, words("tensor cores", size=22, color=GPU, weight=MEDIUM).move_to(core_box))
        small = self.product[0][0].copy()
        self.say("The tensor cores sit at the end of a chain.",
                 FadeOut(self.product), Derive(small, core), run_time=2.0)
        self.read()

        q = Rectangle(width=0.5, height=2.2).set_fill(CPU, 0.35).set_stroke(CPU, 1.5)
        k = Rectangle(width=0.5, height=2.2).set_fill(CPU, 0.35).set_stroke(CPU, 1.5)
        pos = Rectangle(width=0.85, height=2.2).set_fill(TERM_C2P, 0.3).set_stroke(TERM_C2P, 1.5)
        stores = VGroup(q, k, pos).arrange(RIGHT, buff=0.28).move_to(memory[0]).shift(UP * 0.12)
        names = VGroup(*(words(n, size=17, color=INK_DIM).next_to(m, DOWN, buff=0.08)
                         for n, m in zip(("Q", "K", "positions"), stores)))
        self.say("The words' vectors, Q and K, and the position vectors live in GPU memory: plenty of room, but far away.",
                 FadeIn(memory), LaggedStart(*(GrowFromEdge(s, DOWN) for s in stores), lag_ratio=0.2), FadeIn(names),
                 run_time=2.4)
        self.read()
        self.say("Next to the tensor cores is a small, fast on-chip memory. Tiles must be there when the cores want them.",
                 FadeIn(chip_mem))
        self.read()
        self.board = VGroup(memory, chip_mem, core, stores, names)
        self.memory, self.chip_mem, self.core_box, self.core_label, self.stores = memory, chip_mem, core_box, core[1], stores

    # 3 ------------------------------------------------------------------------------------------------------------
    def before(self):
        self.chapter(3, 4, "Before: starved")
        stores, core_box = self.stores, self.core_box
        rng = np.random.default_rng(3)
        reads = VGroup()
        for _ in range(9):
            for source in stores:
                start = source.get_center() + np.array([rng.uniform(-0.2, 0.2), rng.uniform(-1.1, 1.1), 0])
                end = core_box.get_left() + np.array([0, rng.uniform(-0.6, 0.6), 0])
                reads.add(Line(start, end).set_stroke(source.get_stroke_color(), 1.4, opacity=0.55))
        self.say("The previous kernel fetched Q, K and a position vector separately for each pair of words.",
                 LaggedStart(*(ShowPassingFlash(r.copy().set_stroke(width=3, opacity=1), time_width=0.6) for r in reads),
                             lag_ratio=0.06), LaggedStart(*(Create(r) for r in reads), lag_ratio=0.06), run_time=3.0)
        self.read()
        idle = words("mostly idle", size=18, color=INK_DIM).next_to(core_box, DOWN, buff=0.15)
        self.say("Small scattered reads from far away. The tensor cores spent most of their time waiting.",
                 core_box.animate.set_stroke(WASTE, 2.5), self.core_label.animate.set_color(WASTE), FadeIn(idle))
        self.read()

        self.rate = ValueTracker(0)
        rate = counter(self.rate, size=40, color=NUMBER, fmt="{:.1f}")
        unit = words("TFLOP/s", size=20, color=INK_DIM)
        rate.add_updater(lambda m: m.next_to(idle, DOWN, buff=0.3).shift(LEFT * 0.55))
        unit.add_updater(lambda m: m.next_to(rate, RIGHT, buff=0.12).align_to(rate, DOWN))
        rate.update(), unit.update()
        compare = words("plain attention on the same T4: 17", size=16, color=INK_DIM).next_to(unit, DOWN, buff=0.3)
        compare.match_x(core_box)
        self.add(rate)
        self.say("Measured: 2.7 trillion operations a second, where plain attention on the same T4 reaches 17.",
                 FadeIn(unit), self.rate.animate.set_value(2.7), FadeIn(compare), run_time=2.4)
        self.read(extra=0.5)
        self.beat(FadeOut(VGroup(reads, idle, rate, unit, compare)), core_box.animate.set_stroke(GPU, 2.5),
                  self.core_label.animate.set_color(GPU), run_time=1.0)
        self.remove(rate)

    # 4 ------------------------------------------------------------------------------------------------------------
    def now(self):
        self.chapter(4, 4, "Now: load once, reuse")
        memory, chip_mem, core_box, stores = self.memory, self.chip_mem, self.core_box, self.stores
        slices = VGroup(*(Rectangle(width=m.width, height=0.6).set_fill(m.get_fill_color(), 0.9).set_stroke(width=0)
                          .move_to(m.get_top() + DOWN * 0.45) for m in stores))
        landed = VGroup(*(s.copy() for s in slices)).arrange(RIGHT, buff=0.15).move_to(chip_mem[0]).shift(UP * 0.75)
        load = Arrow(memory[0].get_right(), chip_mem[0].get_left(), buff=0.12, stroke_width=5,
                     max_tip_length_to_length_ratio=0.2).set_color(INK).shift(UP * 0.75)
        self.say("Our kernel takes a whole tile of 32 words at once: one wide, contiguous read.",
                 FadeIn(slices), run_time=1.4)
        self.beat(GrowArrow(load), *(Derive(s, l) for s, l in zip(slices, landed)), run_time=2.0)
        self.read()
        window = words("only the distances\nthis tile needs", size=16, color=TERM_C2P).next_to(landed, DOWN, buff=0.12)
        self.say("For positions it loads only the distances that occur between these two tiles, not the whole table.",
                 FadeIn(window, shift=0.1 * UP))
        self.read()

        feed = Arrow(chip_mem[0].get_right(), core_box.get_left(), buff=0.1, stroke_width=5,
                     max_tip_length_to_length_ratio=0.15).set_color(GPU)
        minis = VGroup(*(tile(4, 2, GPU, cell=0.09, gap=0.015).move_to(landed.get_center()) for _ in range(10)))
        self.say("Then it cuts the tile into the 16-by-8 pieces the tensor cores take, and reuses it for every pair.",
                 GrowArrow(feed), core_box.animate.set_fill(GPU, 0.25),
                 LaggedStart(*(Succession(FadeIn(m, run_time=0.15),
                                          m.animate(run_time=0.55).move_to(core_box.get_center()).set_opacity(0))
                               for m in minis), lag_ratio=0.12), run_time=2.8)
        self.read()

        prefetch = Arrow(memory[0].get_right(), chip_mem[0].get_left(), buff=0.12, stroke_width=3,
                         max_tip_length_to_length_ratio=0.2).set_color(INK_DIM).shift(DOWN * 0.9)
        next_tile = VGroup(*(Rectangle(width=m.width, height=0.6).set_fill(m.get_fill_color(), 0.5).set_stroke(width=0)
                             .move_to(m.get_center() + DOWN * 0.2) for m in stores))
        note = words("the T4 has no asynchronous copy: the kernel overlaps by hand", size=16, color=INK_FAINT)
        note.next_to(VGroup(memory[0], chip_mem[0]), DOWN, buff=0.3).align_to(memory[0], LEFT)
        busy = words("busy", size=18, color=GPU).next_to(core_box, DOWN, buff=0.15)
        minis2 = VGroup(*(tile(4, 2, GPU, cell=0.09, gap=0.015).move_to(landed.get_center()) for _ in range(10)))
        self.say("While the cores work, the next tile is already on its way into registers, so they never wait.",
                 FadeIn(next_tile), GrowArrow(prefetch), FadeIn(note), FadeIn(busy),
                 LaggedStart(*(Succession(FadeIn(m, run_time=0.15),
                                          m.animate(run_time=0.55).move_to(core_box.get_center()).set_opacity(0))
                               for m in minis2), lag_ratio=0.12), run_time=2.8)
        self.read()

        self.beat(FadeOut(VGroup(self.board, slices, landed, load, window, feed, prefetch, next_tile, note, busy)),
                  run_time=1.0)
        chart_title = words("attention per layer, 16 × 12 heads × 512 tokens, Tesla T4", size=20, color=INK_DIM)
        unit = 6.0 / 7.0
        rows = VGroup()
        for k, (label, value, color, text) in enumerate([("before: Triton kernel + position tables", 7.0, WASTE, "7.0 ms"),
                                                          ("glinert4 kernel", 2.84, GPU, "2.84 ms")]):
            y = 0.6 - k * 0.7
            bar = Rectangle(width=value * unit, height=0.38).set_fill(color, 0.9).set_stroke(width=0)
            bar.move_to([-1.2, y, 0], aligned_edge=LEFT)
            name = words(label, size=20, color=INK).move_to([-1.5, y, 0], aligned_edge=RIGHT)
            number = words(text, size=20, color=NUMBER if color == GPU else INK, weight=BOLD, font=MONO)
            rows.add(VGroup(name, bar, number.next_to(bar, RIGHT, buff=0.2)))
        chart_title.next_to(rows, UP, buff=0.3).align_to(rows, LEFT)
        self.say("Before: 7.0 milliseconds of attention per layer, counting the position tables built beforehand.",
                 FadeIn(chart_title), FadeIn(rows[0][0]), GrowFromEdge(rows[0][1], LEFT), FadeIn(rows[0][2]))
        self.read()
        gain = chip("2.5× faster", NUMBER, size=22).next_to(rows[1][2], RIGHT, buff=0.35)
        self.say("With the tiles re-packed: 2.84 milliseconds. The same arithmetic, two and a half times faster.",
                 FadeIn(rows[1][0]), GrowFromEdge(rows[1][1], LEFT), FadeIn(rows[1][2]), run_time=2.0)
        self.beat(FadeIn(gain, scale=1.15), run_time=0.8)
        self.read()

In [ ]:
video = render("clip_e_tiles.py", "Tiles")
gif(video, "clip_e_tiles.gif")